# Clase 151 — vLLM / TGI: serving LLM en producción

Simulamos **continuous batching** y **PagedAttention** con asyncio + colas. La API real de vLLM se muestra en markdown.

In [ ]:
import numpy as np, asyncio, time, random
from collections import deque
random.seed(42); np.random.seed(42)

## 1. API de vLLM (conceptual)

```python
from vllm import LLM, SamplingParams
llm = LLM(model='mistralai/Mistral-7B-Instruct-v0.2')
params = SamplingParams(temperature=0.7, max_tokens=128)
outputs = llm.generate(['Hola, ¿quién sos?', 'Explica RAG'], params)
for o in outputs: print(o.outputs[0].text)
```

vLLM aporta: **continuous batching** (no esperar a que termine el batch), **PagedAttention** (KV cache en bloques), throughput 5-20× vs HF transformers naive.

## 2. Simular naive batching (batch fijo)

Generamos N requests con longitudes variables. En batching naive, todos esperan al request más largo.

In [ ]:
def gen_requests(n=24):
    return [{'id': i, 'in_len': random.randint(20, 60),
             'out_len': random.randint(20, 200)} for i in range(n)]

TOKEN_TIME = 0.001   # 1 ms/token (simulado, escala libre)

def naive_batched(requests, batch_size=8):
    """Batch fijo: el batch entero termina cuando termina el más largo."""
    t0 = time.perf_counter(); total_tokens = 0; latencies = []
    for i in range(0, len(requests), batch_size):
        batch = requests[i:i+batch_size]
        max_len = max(r['out_len'] for r in batch)
        time.sleep(max_len * TOKEN_TIME)   # bloquea hasta el más largo
        for r in batch:
            total_tokens += r['out_len']
            latencies.append(max_len * TOKEN_TIME)
    dur = time.perf_counter() - t0
    return dur, total_tokens / dur, latencies

reqs = gen_requests(24)
dur_n, tps_n, lat_n = naive_batched(reqs, batch_size=8)
print(f'NAIVE: {dur_n*1000:6.1f} ms | throughput={tps_n:7.1f} tok/s | p50={np.percentile(lat_n,50)*1000:.1f}ms p99={np.percentile(lat_n,99)*1000:.1f}ms')

## 3. Continuous batching con asyncio

Cada step del scheduler: los requests que terminaron salen, entran nuevos. Nadie espera al más largo.

In [ ]:
async def continuous_batching(requests, max_concurrent=8):
    queue = deque(requests); active = []   # active = [(req, tokens_done, start)]
    completed = []; t0 = time.perf_counter()
    while queue or active:
        while len(active) < max_concurrent and queue:
            r = queue.popleft(); active.append([r, 0, time.perf_counter()])
        # 1 step = 1 token por slot activo
        await asyncio.sleep(TOKEN_TIME)
        still = []
        for slot in active:
            slot[1] += 1
            if slot[1] >= slot[0]['out_len']:
                completed.append((slot[0], time.perf_counter() - slot[2]))
            else:
                still.append(slot)
        active = still
    dur = time.perf_counter() - t0
    total_tok = sum(r['out_len'] for r in requests)
    lats = [l for _, l in completed]
    return dur, total_tok / dur, lats

dur_c, tps_c, lat_c = asyncio.run(continuous_batching(reqs, max_concurrent=8))
print(f'CONT.: {dur_c*1000:6.1f} ms | throughput={tps_c:7.1f} tok/s | p50={np.percentile(lat_c,50)*1000:.1f}ms p99={np.percentile(lat_c,99)*1000:.1f}ms')
print(f'\nspeedup: {dur_n/dur_c:.2f}x | throughput gain: {tps_c/tps_n:.2f}x')

## 4. PagedAttention: KV cache en bloques

Inspirado en **paging** del SO. Cada secuencia ocupa bloques de tamaño fijo (típico 16 tokens). Sin paging, hay que pre-reservar `max_seq_len` por request → fragmentación interna brutal.

In [ ]:
class PagedKVCache:
    def __init__(self, total_blocks=64, block_size=16):
        self.bs = block_size
        self.free = list(range(total_blocks))
        self.alloc = {}   # req_id -> [block_ids]
    def append(self, req_id, n_tokens):
        if req_id not in self.alloc: self.alloc[req_id] = []
        cur = len(self.alloc[req_id]) * self.bs
        needed_blocks = max(0, -(-(cur + n_tokens) // self.bs) - len(self.alloc[req_id]))
        for _ in range(needed_blocks):
            if not self.free: raise RuntimeError('OOM en KV cache')
            self.alloc[req_id].append(self.free.pop(0))
    def free_req(self, req_id):
        for b in self.alloc.pop(req_id, []): self.free.append(b)
    def utilization(self):
        used = sum(len(v) for v in self.alloc.values())
        total = used + len(self.free)
        return used / total

cache = PagedKVCache(total_blocks=64, block_size=16)
for rid, ntok in [(0, 30), (1, 100), (2, 50)]:
    cache.append(rid, ntok)
    print(f'append req {rid} ({ntok} tok): util={cache.utilization():.2%}')
cache.free_req(1)
print(f'free req 1: util={cache.utilization():.2%} (los bloques vuelven al pool)')

## 5. Comparativa de waste de memoria

In [ ]:
max_len = 2048; block = 16
actual_lens = [random.randint(50, 600) for _ in range(20)]
naive_kv = len(actual_lens) * max_len
paged_kv = sum(-(-l // block) * block for l in actual_lens)   # ceil al bloque
actual = sum(actual_lens)
print(f'tokens reales:      {actual:6,}')
print(f'pre-reserva naive:  {naive_kv:6,} → waste={1-actual/naive_kv:.1%}')
print(f'paged:              {paged_kv:6,} → waste={1-actual/paged_kv:.1%}')

## 6. TGI vs vLLM

| | vLLM | TGI (HuggingFace) |
|--|--|--|
| Continuous batching | ✅ | ✅ |
| PagedAttention | ✅ (inventor) | parcial |
| Streaming | ✅ | ✅ |
| Multi-LoRA | ✅ | ✅ |
| Quantization | AWQ/GPTQ/FP8 | bitsandbytes/EETQ |
| Speculative decoding | ✅ | ✅ |

## Ejercicio guiado

1. Subir `max_concurrent` ∈ {4, 8, 16, 32} y graficar throughput.
2. Distribuir requests con burst (todos llegan a la vez) vs Poisson.
3. Simular OOM cuando todos los requests son largos: ¿qué hace el scheduler?

## Conclusiones

- Continuous batching duplica throughput sin tocar el modelo.
- PagedAttention reduce waste de KV cache de ~80% a ~5%.
- En producción: vLLM por throughput puro; TGI si ya estás en stack HF.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README de esta clase. El código que usa librerías pesadas (`transformers` / `torch` / `keras` / `diffusers`) es la **API real** de la industria y se valida por sintaxis (los modelos requieren GPU/descarga). Los **núcleos numéricos** están en numpy puro, son **ejecutables** y se autoverifican con `assert`.

### Ejercicio 1 — vLLM básico (servidor OpenAI-compatible)

In [ ]:
try:
    import vllm  # noqa: F401
    _VLLM = True
except Exception:
    _VLLM = False
if _VLLM:
    from vllm import LLM, SamplingParams
    llm = LLM(model='mistralai/Mistral-7B-Instruct-v0.2')
    print(llm.generate(['Hola, ¿quién sos?'],
                       SamplingParams(temperature=0.7, max_tokens=64))[0].outputs[0].text)
else:
    print('CLI: python -m vllm.entrypoints.openai.api_server '
          '--model mistralai/Mistral-7B-Instruct-v0.2  (cliente OpenAI apuntando a localhost).')

### Ejercicio 2 — Continuous batching vs naive (ejecutable, sin asyncio)

In [ ]:
# Núcleo ejecutable: contamos PASOS (1 token/slot activo) para procesar los
# mismos requests con batch fijo (naive) vs continuous batching.
from collections import deque
def naive_steps(reqs, bs):
    s = 0
    for i in range(0, len(reqs), bs):
        s += max(r['out_len'] for r in reqs[i:i+bs])   # todos esperan al mas largo
    return s
def cont_steps(reqs, mc):
    q = deque(reqs); active = []; steps = 0
    while q or active:
        while len(active) < mc and q:                  # entran nuevos al liberarse slots
            active.append([q.popleft(), 0])
        steps += 1
        active = [[r, t + 1] for r, t in active if t + 1 < r['out_len']]
    return steps
ns_, cs_ = naive_steps(reqs, 8), cont_steps(reqs, 8)
print(f'naive={ns_} pasos | continuous={cs_} pasos | speedup={ns_/cs_:.2f}x')
assert cs_ < ns_        # continuous batching termina en menos pasos

### Ejercicio 3 — AWQ quantization: VRAM ~5 GB vs 14 GB fp16

In [ ]:
if _VLLM:
    from vllm import LLM
    llm = LLM(model='TheBloke/Mistral-7B-Instruct-v0.2-AWQ', quantization='awq')
    print('AWQ 4-bit: ~5 GB VRAM (vs ~14 GB fp16), calidad casi intacta.')
else:
    print('AWQ (Activation-aware Weight Quantization): 4-bit, ~5 GB para un 7B.')

### Ejercicio 4 — Structured JSON output (guided decoding)

In [ ]:
schema = {'type': 'object',
          'properties': {'sentiment': {'type': 'string',
                                       'enum': ['pos', 'neg', 'neu']},
                         'score': {'type': 'number'}},
          'required': ['sentiment', 'score']}
if _VLLM:
    # cliente OpenAI -> extra_body={'guided_json': schema} fuerza JSON valido
    print('extra_body={"guided_json": schema} -> el decoder solo emite tokens'
          ' que respetan el JSON Schema.')
else:
    import json
    print('JSON Schema forzado:', json.dumps(schema))

### Ejercicio 5 — TGI (Text Generation Inference)

In [ ]:
print('docker run --gpus all -p 8080:80 '
      'ghcr.io/huggingface/text-generation-inference:latest '
      '--model-id mistralai/Mistral-7B-Instruct-v0.2')
print('TGI: continuous batching + streaming; ideal si ya estás en stack HuggingFace.')